## Description
Write a program to perform motion estimation on a provided video sequence. You will have to implement three search algorithms to find the motion vectors, the Full Search method, 2D Logarithmic Search method and Three-Step Search method. For the methods, estimate motion vectors for non-overlapping macroblocks within a search window of ±p pixels in both the horizontal and vertical directions. Use the Sum of Absolute Differences (SAD), which is described on the class materials, as the metric for block matching.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math
import os

In [ ]:
from pathlib import Path
ROOT_DIR = '.'
Path(ROOT_DIR, 'output').mkdir(parents=True, exist_ok=True)

# Show reference images
You should see 2 images

In [ ]:
ref_img = cv2.imread(ROOT_DIR+ '/img/008.jpg')
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
ref_img = np.array(ref_img)
plt.imshow(ref_img)
plt.show()

target_img = cv2.imread(ROOT_DIR+ '/img/009.jpg')
target_img = cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB)
target_img = np.array(target_img)
plt.imshow(target_img)
plt.show()

#Implement the full search algorithm


In [ ]:
def full_search(target_img, ref_img, search_range, block_size):
  """
  Inputs:
    target_img: the frame to match.
    ref_img: the reference frame to search within.
    search_range: how far (in pixels) the algorithm searches around each block position.
    block_size: the size of each square block.
  """
  h, w, _ = target_img.shape # height and width of image
  result_img = np.zeros_like(target_img) # fill with 0s to put the best-matched image

  # coordinates for motion vectors
  start_point = []
  end_point = []

  total_SAD = 0 # total matching error

  #TODO #1 (5%)
  #Finds, for every block in target_img,
  #the most similar block in ref_img within the defined search_range
  #using the SAD as the similarity measure.

  #TODO #2 (3%)
  #Describe the core logic of you code

  # loop through each block in the target image
  for y in range(0, h, block_size):
    for x in range(0, w, block_size):
      target_block = target_img[y:y+block_size, x:x+block_size] # current block from target frame

      # init best match
      min_sad = float('inf')
      best_match_coord = (y, x)

      # define search window boundaries in reference image
      y_min = max(0, y - search_range)
      y_max = min(h - block_size, y + search_range)
      x_min = max(0, x - search_range)
      x_max = min(w - block_size, x + search_range)

      # search all candidate blocks
      for ry in range(y_min, y_max + 1):
        for rx in range(x_min, x_max + 1):
          ref_block = ref_img[ry:ry+block_size, rx:rx+block_size]

          # compute SAD (diff between blocks)
          sad = np.sum(np.abs(target_block.astype(np.int32) - ref_block.astype(np.int32)))

          # update best match
          if sad < min_sad:
            min_sad = sad
            best_match_coord = (ry, rx)

      total_SAD += min_sad # add best SAD to total error
      # copy best matching block to result image
      ry_best, rx_best = best_match_coord
      result_img[y:y+block_size, x:x+block_size] = ref_img[ry_best:ry_best+block_size, rx_best:rx_best+block_size]

      # store motion vectors (start = target center, end = ref center)
      start_point.append((x + block_size // 2, y + block_size // 2))
      end_point.append((rx_best + block_size // 2, ry_best + block_size // 2))
  # store motion vector
  start_point = np.array(start_point)
  end_point = np.array(end_point)

    # Core logic:
    # 1. take one target block
    # 2. search all candidate blocks in ref frame inside ±p window
    # 3. compute SAD for each candidate
    # 4. keep the block with smallest SAD
    # 5. copy best block to result image and save motion vector

  """
  Outputs:
    result_img: reconstructed image using best-matched blocks.
    start_point & end_point: coordinates of motion vectors.
    total_SAD: total sum of absolute differences (measure of overall match quality).
  """
  return result_img, start_point, end_point, total_SAD

#Implement PSNR
You can implment later and left it blank.

In [ ]:
def PSNR(img,new_img, new_img_filename=""):
    psnr = 0
    #TODO #7 (2%)
    #Implement PSNR

    #TODO #8 (1%)
    #Comment your code
    psnr = 0

    # PSNR uses MSE first, then 10*log10(MAX^2 / MSE)
    img = img.astype(np.float64)
    new_img = new_img.astype(np.float64)

    mse = np.mean((img - new_img) ** 2) # the mean is for 1/mn and the ∑∑

    # if images are identical, PSNR is infinite (try inserting when MSE = 0 since identical)
    if mse == 0:
        psnr = float('inf')
    else:
        MAX = 255.0 # since img is represented by 8 bits
        psnr = 10 * np.log10((MAX ** 2) / mse) # formula

    if new_img_filename:
        print(f'PSNR of the image {new_img_filename} is {psnr}')
    else:
        print('PSNR of the image is',psnr)
    return psnr

#Show the results of the full search method.
* Each image receives a 1% increase, for a total of 12%.
* 5-1. Execute the full search method
  * List the search range, block size
  * Show image
  * Print SAD
  * Evaluate PSNR
    <br>You can check after you finish implement PSNR
  * Store the image to output directory
* 5-2. Motion vector of the full search method
  * List the search range, block size
  * Store the image to output directory
* 5-3. Residual of the full search method
  * List the search range, block size
  * Store the image to output directory


In [ ]:
def execute_full_search(ref_img, target_img, search_range, block_size) :
    # full search method
    print("\nOutput the full search method on search_range=", search_range, ", block_size=", block_size)
    full_search_img, start_point, end_point, total_SAD = full_search(target_img, ref_img, search_range, block_size)
    output_name = 'full_search_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(output_name)
    plt.imshow(full_search_img)
    plt.show()
    #print PSNR
    PSNR(target_img, full_search_img, output_name)
    #print SAD
    print('total SAD: ' + str(total_SAD))
    output_img = cv2.cvtColor(full_search_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + output_name, output_img)

    # motion vector img of full search method
    print("\nMotion vector of the full search method on search_range=", search_range, ", block_size=", block_size)
    motion_img = np.copy(target_img)
    for i in range(start_point.shape[0]):
        motion_img = cv2.arrowedLine(motion_img, start_point[i], end_point[i], (255,0,0),1,tipLength = 0.2)
    motion_img_name = 'full_search_motion_vector_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(motion_img_name)
    plt.imshow(motion_img)
    plt.show()
    output_img = cv2.cvtColor(motion_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + motion_img_name, output_img)

    # residual img of full search method
    print("\nResidual of the full search method on search_range=", search_range, ", block_size=", block_size)
    residual_img = np.abs(np.subtract(target_img.astype(int), full_search_img.astype(int)))
    residual_img_name = 'full_search_residual_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(residual_img_name)
    plt.imshow(residual_img)
    plt.show()
    output_img = cv2.cvtColor(residual_img.astype('uint8'), cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + residual_img_name, output_img)


In [ ]:
#Execute and show the images, each setting has 3 images
execute_full_search(ref_img, target_img, 8,8)
execute_full_search(ref_img, target_img, 8,16)
execute_full_search(ref_img, target_img, 16,8)
execute_full_search(ref_img, target_img, 16,16)

#[TODO #4] Report of the full search method (4%)
Compare and discuss the motion-based image prediction quality for all the above settings

Your repoert here.

I tested four settings with different search ranges and block sizes.

From the results:

*   r=8, b=8 → PSNR = 29.08, SAD = 9,580,644
*   r=8, b=16 → PSNR = 27.93, SAD = 11,039,398
*   r=16, b=8 → PSNR = 31.09, SAD = 8,529,009
*   r=16, b=16 → PSNR = 29.47, SAD = 10,102,414

**The best result is r=16, b=8** because it has the **highest PSNR** and **lowest SAD**. I think that the residual image should also be darker, indicating that the prediction is closer to the target frame, but I can't really see the difference visually from the laptop screen. But when sending it to my phone and increasing the phone's brightness, I can confirm that the r=16, b=8 has the darkest residual image.

The worst one is r=8, b=16. The PSNR is the lowest and the residual is more visible, so the matching is not as good.

From what I see, **smaller blocks (8×8) work better than 16×16**. This is probably because smaller blocks can follow the motion of the person more accurately. With bigger blocks, one block may include both the person and the background, so it’s harder to match well.

Also, **increasing the search range from 8 to 16 improves the result**. With a larger search area, the algorithm has more choices to find a better match.

Looking at the motion vector images, most of the strong motion vectors are around the person, which makes sense because the person is moving. The background mostly has small or noisy vectors.

Overall, r=16 and b=8 gives the best quality, since it matches the motion better and reduces the error the most.

#Implement the 2D logarithmic search algorithm

In [ ]:
def log_search(target_img, ref_img, search_range, block_size):
  """
  Input:
    target_img: the image (or frame) whose motion is being tracked.
    ref_img: the reference image to search within.
    search_range: the maximum displacement allowed.
    block_size: the dimension of each square block.
  """
  h, w, _ = target_img.shape
  result_img = np.zeros_like(target_img)

  start_point = []
  end_point = []
  total_SAD = 0

  def get_sad(target_block, ry, rx):
    # D(i, j): matching error function using SAD.
    ref_block = ref_img[ry:ry + block_size, rx:rx + block_size]
    return np.sum(np.abs(target_block.astype(np.int32) - ref_block.astype(np.int32)))

  # start at center, check 5 points, move to best point, then halve step
  for y in range(0, h - block_size + 1, block_size):
    for x in range(0, w - block_size + 1, block_size):
      target_block = target_img[y:y + block_size, x:x + block_size]

      # step 1: initialization
      q, l = 0, 0

      # n' = floor(log2 p)
      if search_range > 0:
        n_prime = math.floor(math.log2(search_range))
        n = max(2, 2 ** (n_prime - 1))
      else:
        n = 1 #search range = 0

      # main logarithmic search
      while n > 1:
        # Step 2: M'(n) = {(0,0), (n,0), (0,n), (-n,0), (0,-n)}
        candidates = [
        (0, 0), (n, 0), (0, n), (-n, 0), (0, -n)
        ]

        best_i, best_j = 0, 0
        best_sad = float("inf") # start w worst possible error

        # step 3: find (i, j) in M'(n) that minimizes D(i+q, j+l)
        for i, j in candidates:
          ry = y + l + j
          rx = x + q + i

          # must stay inside image
          if not (0 <= ry <= h - block_size and 0 <= rx <= w - block_size):
            continue

          # must stay inside search window N(p), if not skip
          if abs(q + i) > search_range or abs(l + j) > search_range:
            continue

          sad = get_sad(target_block, ry, rx) # compute error

          if sad < best_sad: # compare with best so far
            best_sad = sad # update if better
            best_i, best_j = i, j

        # if best point is center (i=0 j=0), go to Step 5
        if best_i == 0 and best_j == 0:
          n = n // 2
        else:
          # step 4: move the center, keep same step size
          q += best_i
          l += best_j

      # step 6: final search in N(1)
      final_best_q, final_best_l = q, l
      final_best_sad = float("inf")

      for j in [-1, 0, 1]:
        for i in [-1, 0, 1]:
          cand_q = q + i
          cand_l = l + j

          ry = y + cand_l
          rx = x + cand_q

          if not (0 <= ry <= h - block_size and 0 <= rx <= w - block_size):
            continue

          if abs(cand_q) > search_range or abs(cand_l) > search_range:
            continue

          sad = get_sad(target_block, ry, rx)

          if sad < final_best_sad:
            final_best_sad = sad
            final_best_q = cand_q
            final_best_l = cand_l

      # copy matched block
      match_y = y + final_best_l
      match_x = x + final_best_q

      result_img[y:y + block_size, x:x + block_size] = \
      ref_img[match_y:match_y + block_size, match_x:match_x + block_size]

      total_SAD += final_best_sad

      # motion vector: original block center -> matched block center
      start_point.append((x + block_size // 2, y + block_size // 2))
      end_point.append((match_x + block_size // 2, match_y + block_size // 2))
  start_point = np.array(start_point)
  end_point = np.array(end_point)

# Core logic:
# 1. start from the current block position
# 2. check 5 points: center, up, down, left, and right
# 3. find the point with the minimum SAD
# 4. if the center is the best point, reduce the step size by half
# 5. otherwise, move the search center to that best point
# 6. repeat until the step size becomes 1
# 7. do a final 3x3 local search around the best point

  """
  Outputs:
    result_img: reconstructed image from matched reference blocks.
    start_point and end_point: motion vector coordinates.
    total_SAD: overall measure of block matching error.
  """
  return result_img, start_point, end_point, total_SAD

#Show the results of the 2D logarithmic search method
* Each image receives a 1% increase, for a total of 12%.
* 6-1. Execute the 2D logarithmic search method
  * List the search range, block size
  * Show image
  * Print SAD
  * Evaluate PSNR
  * Store the image to output directory
* 6-2. Motion vector of the 2D logarithmic search
  * List the search range, block size
  * Store the image to output directory
* 6-3. Residual of 2D logarithmic search
  * List the search range, block size
  * Store the image to output directory


In [ ]:
def execute_2d(ref_img, target_img, search_range, block_size) :

    log_img, start_point, end_point, total_SAD = log_search(target_img, ref_img, search_range, block_size)
    output_name = '2d_predicted_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(output_name)
    plt.imshow(log_img)
    plt.show()
    PSNR(target_img, log_img)
    print('total SAD of the above image is ' + str(total_SAD))
    output_img = cv2.cvtColor(log_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + output_name, output_img)

    # motion vector img of 2D logarithmic search method
    motion_img = np.copy(target_img)
    for i in range(start_point.shape[0]):
        motion_img = cv2.arrowedLine(motion_img, start_point[i], end_point[i], (255,0,0),1,tipLength = 0.2)
    motion_img_name = '2d_motion_vector_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(motion_img_name)
    plt.imshow(motion_img)
    plt.show()
    output_img = cv2.cvtColor(motion_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + motion_img_name, output_img)

    # residual img of full search method
    residual_img = np.abs(np.subtract(target_img.astype(int), log_img.astype(int)))
    residual_img_name = '2d_residual_r'+str(search_range)+'_b'+str(block_size)+'.jpg'
    plt.title(residual_img_name)
    plt.imshow(residual_img)
    plt.show()
    output_img = cv2.cvtColor(residual_img.astype('uint8'), cv2.COLOR_RGB2BGR)
    cv2.imwrite(ROOT_DIR + '/output/' + residual_img_name, output_img)

#Execute
execute_2d(ref_img, target_img, 8,8)
execute_2d(ref_img, target_img, 8,16)
execute_2d(ref_img, target_img, 16,8)
execute_2d(ref_img, target_img, 16,16)

#[TODO #6] Report of 2D logarithmic search method (4%)
Compare and discuss the motion-based image prediction quality for all the above settings

Your repoert here.



I tested four settings with different search ranges and block sizes.

From the results:

*   r=8, b=8 → PSNR = 27.43, SAD = 11,012,705
*   r=8, b=16 → PSNR = 26.79, SAD = 11,934,741
*   r=16, b=8 → PSNR = 29.04, SAD = 10,113,183
*   r=16, b=16 → PSNR = 28.46, SAD = 10,867,854

**The best result is r=16, b=8**, since it has the **highest PSNR** and **lowest SAD**. The residual image is also darker compared to the others, which means the prediction is closer to the target frame.

The worst result is r=8, b=16, where the PSNR is the lowest and the residual is more visible. This means the matching is not as accurate.

Similar to full search, **smaller block size (8×8) gives better results** than 16×16. Smaller blocks can follow the motion of the person more precisely, while larger blocks include more background, making matching harder.

Also, **increasing the search range from 8 to 16 improves the quality**. A larger search area allows the algorithm to find better matching blocks, especially when the motion is larger.

From the motion vector images, the vectors around the person are more noticeable, while the background mostly has small or noisy vectors. This shows that the algorithm is capturing the main motion of the person.

However, **compared to full search, the PSNR values here are slightly lower and SAD is higher**. This is expected because **2D logarithmic search is a faster method that does not check all possible positions**, so it may miss the best match sometimes.

Overall, r=16 and b=8 still gives the best performance for 2D log search, but **the quality is worse than full search** due to fewer search points.

Now, you can go back and start to implement the def PSNR()

#Plot SAD and PSNR
* Compare the SAD and PSNR of the full search method and the 2d logarithmic search.
* Use the search range of p = 8 and macroblock size of 16×16.
* The reference image is 000.jpg, and the target images are the image sequence which starts from 001.jpg to 017.jpg.

In [ ]:
def evaluate_motion_estimation(search_range, block_size):
    psnr_list_full_search = []
    SAD_list_full_search = []
    psnr_list_log_search = []
    SAD_list_log_search = []

    print(f"\nEvaluating motion estimation for search_range={search_range}, block_size={block_size}")

    # Load the reference image once outside the loop
    ref_img_path_base = os.path.join(ROOT_DIR, f'img/000.jpg')
    ref_img_base = cv2.imread(ref_img_path_base)
    if ref_img_base is None:
        print(f"Error: Could not load reference image from {ref_img_path_base}")
        return
    ref_img_base = cv2.cvtColor(ref_img_base, cv2.COLOR_BGR2RGB)
    ref_img_base = np.array(ref_img_base)

    for i in range(18):
        # Use 000.jpg as the reference image, and 00x.jpg as the target image
        ref_img = ref_img_base # Reference image remains 000.jpg

        target_img_path = os.path.join(ROOT_DIR, f'img/{i:03d}.jpg')
        target_img = cv2.imread(target_img_path)

        if target_img is None:
            print(f"Error: Could not load target image from {target_img_path}")
            continue

        target_img = cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB)
        target_img = np.array(target_img)

        # Full Search
        full_search_img, _, _, total_SAD_full = full_search(target_img, ref_img, search_range, block_size)
        psnr_full = PSNR(target_img, full_search_img, f'full_search_r{search_range}_b{block_size}_{i:03d}.jpg')
        psnr_list_full_search.append(psnr_full)
        SAD_list_full_search.append(total_SAD_full)

        # 2D Logarithmic Search
        log_search_img, _, _, total_SAD_log = log_search(target_img, ref_img, search_range, block_size)
        psnr_log = PSNR(target_img, log_search_img, f'log_search_r{search_range}_b{block_size}_{i:03d}.jpg')
        psnr_list_log_search.append(psnr_log)
        SAD_list_log_search.append(total_SAD_log)

    x = list(range(len(psnr_list_full_search)))

    # Plot PSNR for both methods
    plt.figure(figsize=(12, 6))
    plt.plot(x, psnr_list_full_search, marker='o', label='Full Search PSNR')
    plt.plot(x, psnr_list_log_search, marker='x', label='2D Log Search PSNR')
    plt.title(f"PSNR Comparison (Range={search_range}, Block={block_size})")
    plt.xlabel("Frame Index")
    plt.ylabel("PSNR (dB)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot SAD for both methods
    plt.figure(figsize=(12, 6))
    plt.plot(x, SAD_list_full_search, marker='o', label='Full Search SAD')
    plt.plot(x, SAD_list_log_search, marker='x', label='2D Log Search SAD')
    plt.title(f"SAD Comparison (Range={search_range}, Block={block_size})")
    plt.xlabel("Frame Index")
    plt.ylabel("SAD")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

#Evaluate
Just run and explain the results.

In [ ]:
evaluate_motion_estimation(search_range=8, block_size=16)

# You can practice the other settings
evaluate_motion_estimation(search_range=8, block_size=8)
evaluate_motion_estimation(search_range=16, block_size=8)
evaluate_motion_estimation(search_range=16, block_size=16)

#[TODO #9] Explain the evaluation (%4)
* Describe what you observe
* Compare the SAD and PSNR of the full search method and the 2d logarithmic search


From the graphs and the numbers, I can see that **full search always gives better results** than 2D logarithmic search. The PSNR values for full search are consistently higher, and the SAD values are lower. This means full search is finding better matches between blocks.

**At the start (frame 0), both methods give infinite PSNR**, which makes sense because the frames are the same. After that, for search_range = 8 and block_size = 16, the gap is not very big, maybe around 0.3 to 0.7 dB. So in this case, log search is actually pretty close to full search.

When I reduce the block size to 8, I notice that the gap becomes a bit larger. The PSNR from log search drops more compared to full search. I think this is because **smaller blocks are more sensitive**, so if the algorithm doesn’t find the exact best match, the error increases more.

When I increase the search range to 16, the difference becomes much more obvious. Full search improves a lot, but log search doesn’t improve as much. The PSNR gap can go up to around 2–3 dB. At the same time, the SAD for log search is higher than full search in all frames.

**To conclude:**

# **PSNR**
*   For search_range = 8, block_size = 16, the difference is quite small, usually around 0.3 to 0.7 dB, so log search is still pretty close to full search.
*   When I change to block_size = 8, the gap becomes larger, around 0.5 to 1 dB. I think this is because smaller blocks are more sensitive, so even a slightly worse match affects the result more.
*   For search_range = 16, the difference becomes much more obvious. The gap can reach around 2 to 3 dB, which shows that log search is missing better matches more often.


# **SAD**
*   Full search always has lower SAD
*   Log search always has higher SAD

# **OVERALL**
When using smaller blocks (8×8):
*   PSNR is higher overall
*   But the gap between full search and log search becomes larger
this means smaller blocks are more sensitive to motion differences and matching errors.

When increasing the search range (8 → 16):
*   Full search improves a lot
*   Log search improves less\
so the gap between them becomes bigger.

**full search gives the best quality (higher PSNR and lower SAD)** because it checks all possible positions.

**2D logarithmic search is faster but less accurate**, since it only checks a limited number of points and may miss the optimal match.